In [1]:
import jieba
import jieba.analyse
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# 建立詞彙表

以下4個對於影片觀後感言、正或負面評價

In [113]:
texts = [
    "這部電影很好看，劇情精彩，值得一看再看",
    "演技很差，浪費時間，這部電影不值得看",
    "電影畫面雖漂亮，但故事無聊，劇情也不連貫",
    "超棒的電影！故事、劇情俱佳，值得推薦"
]
labels = [1, 0, 0, 1]   # 1:正面, 0:負面

## 中文斷詞(使用jieba)

使用額外詞典my_dict.txt增強斷詞精確度

In [190]:
jieba.load_userdict('./data/my_dict.txt')
def cutter(line):
    words=jieba.cut(line,cut_all=False,HMM=True)
    return ' '.join(words)

tokenized_texts = [cutter(t) for t in texts]
print(tokenized_texts)


['這部 電影 很 好看 ， 劇情 精彩 ， 值得 一看再看', '演技 很差 ， 浪費 時間 ， 這部 電影 不值得 看', '電影 畫面 雖 漂亮 ， 但 故事 無聊 ， 劇情 也 不連貫', '超棒 的 電影 ！ 故事 、 劇情 俱佳 ， 值得 推薦']


## 1-gram 詞彙表、詞頻
1. 將文字文檔集合轉換為詞彙計數矩陣，參數token_pattern=r'(?u)\b\w\w+\b'讓詞彙擷取時須符合條件，  
    (?u)  unicode 模式  
    \b ：word boundary（單字邊界），單字開始或結束的位置，通常成對  
    \w ：一個字元  
    \w+：一個以上字元      
2. 詞彙計數矩陣每列(row)為給予之文本，相對於詞袋每一詞彙的發生次數

In [197]:
count = CountVectorizer(ngram_range=(1,1),token_pattern=r'(?u)\b\w\w+\b')
bow = count.fit_transform(tokenized_texts)  # 適配轉換
print(dict(sorted(count.vocabulary_.items(), key=lambda item: item[1])))    # 詞袋所有詞彙依次排序
# pd.DataFrame(bow.toarray(),columns=count.get_feature_names_out())
from tabulate import tabulate
print(tabulate(pd.DataFrame(bow.toarray()), 
               headers=sorted(count.vocabulary_.values()), 
               tablefmt='fancy_grid'))

{'一看再看': 0, '不值得': 1, '不連貫': 2, '俱佳': 3, '值得': 4, '劇情': 5, '好看': 6, '很差': 7, '推薦': 8, '故事': 9, '時間': 10, '浪費': 11, '漂亮': 12, '演技': 13, '無聊': 14, '畫面': 15, '精彩': 16, '超棒': 17, '這部': 18, '電影': 19}
╒════╤═════╤═════╤═════╤═════╤═════╤═════╤═════╤═════╤═════╤═════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╕
│    │   0 │   1 │   2 │   3 │   4 │   5 │   6 │   7 │   8 │   9 │   10 │   11 │   12 │   13 │   14 │   15 │   16 │   17 │   18 │   19 │
╞════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╡
│  0 │   1 │   0 │   0 │   0 │   1 │   1 │   1 │   0 │   0 │   0 │    0 │    0 │    0 │    0 │    0 │    0 │    1 │    0 │    1 │    1 │
├────┼─────┼─────┼─────┼─────┼─────┼─────┼─────┼─────┼─────┼─────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┤
│  1 │   0 │   1 │   0 │   0 │   0 │   0 │   0 │   1 │   0 │   0 │    1 │    1 │    0 │    1 │    0 │    0 │    0 │    0

## 使用TfidfVectorizer建立N-gram 詞彙表、TF-IDF
指定ngram_range=(1,1)，等同建構同上1-gram的詞袋模型，同時計算各詞彙在各文件的TF-IDF權值

In [193]:
np.set_printoptions(precision=2,linewidth=200)
vectorizer = TfidfVectorizer(ngram_range=(1,1), max_features=100)
X = vectorizer.fit_transform(tokenized_texts)
vocabulary={k: int(v) for k, v in vectorizer.vocabulary_.items()}
print(dict(sorted(vocabulary.items(), key=lambda item: item[1])))
print(tabulate(pd.DataFrame(X.toarray()), headers=sorted(vocabulary.values()),floatfmt='.2f', tablefmt='fancy_grid'))

{'一看再看': 0, '不值得': 1, '不連貫': 2, '俱佳': 3, '值得': 4, '劇情': 5, '好看': 6, '很差': 7, '推薦': 8, '故事': 9, '時間': 10, '浪費': 11, '漂亮': 12, '演技': 13, '無聊': 14, '畫面': 15, '精彩': 16, '超棒': 17, '這部': 18, '電影': 19}
╒════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╤══════╕
│    │    0 │    1 │    2 │    3 │    4 │    5 │    6 │    7 │    8 │    9 │   10 │   11 │   12 │   13 │   14 │   15 │   16 │   17 │   18 │   19 │
╞════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╪══════╡
│  0 │ 0.45 │ 0.00 │ 0.00 │ 0.00 │ 0.36 │ 0.29 │ 0.45 │ 0.00 │ 0.00 │ 0.00 │ 0.00 │ 0.00 │ 0.00 │ 0.00 │ 0.00 │ 0.00 │ 0.45 │ 0.00 │ 0.36 │ 0.24 │
├────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┼──────┤
│  1 │ 0.00 │ 0.41 │ 0.00 │ 0.00 │ 0.00 │ 0.00 │ 0.00 │ 0.41 │ 0.00 │ 

# 利用上述1-gram詞袋模型的結果，計算TF-IDF權重

In [198]:
n=len(texts) # 文件數
def tf_td(t,d):     # d 文件t的詞頻
    return bow.getrow(d).toarray()[0][t]
def tf_d(d):        # d 文件內的詞頻
    return bow.getrow(d).toarray()[0]
def df(t):          # 包含t的文件數
    return np.count_nonzero(bow.getcol(t).toarray()[:,0])
def idf_t(t):       # t的逆文件頻率
    return np.log((1+n)/(1+df(t)))+1
def tf_idf_d(d):    # d文件 tf-idf 原始值
    return tf_d(d)*[idf_t(t) for t in sorted(count.vocabulary_.values())]
def l2_tf_idf_d(d): # L2 歸一化 tf-idf
    return tf_idf_d(d)/np.sqrt(sum(tf_idf_d(d)**2))

print(l2_tf_idf_d(0))   # 第1筆文本各詞彙的tf-idf權重

[0.45 0.   0.   0.   0.36 0.29 0.45 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.45 0.   0.36 0.24]
